# LeetCode 3464 — Maximize the Distance Between Points on a Square bằng Python

- Link: https://leetcode.com/problems/maximize-the-distance-between-points-on-a-square/
- Độ khó: **Hard**
- Chủ đề: **Duỗi biên hình vuông thành đường tròn** (tọa độ chu vi), **tìm kiếm nhị phân đáp án**, **tham lam chọn điểm kế tiếp** bằng `bisect`, xử lý **vòng tròn**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxDistance(self, side: int, points: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Đưa điểm trên biên hình vuông về **một số** `t ∈ [0, 4·side)` (đi theo chiều kim đồng hồ… hoặc ngược, miễn nhất quán).
2. Chứng minh: với `k ≥ 4` đáp án `≤ side`, và với `d ≤ side` thì "Manhattan ≥ d" giữa các điểm được chọn tương đương "khoảng cách cung ≥ d" giữa các điểm **liền kề trên vòng**.
3. Tìm kiếm nhị phân trên `d` với hàm kiểm tra đơn điệu.
4. Hàm kiểm tra: từ mỗi điểm xuất phát, tham lam nhảy tới điểm gần nhất cách ≥ `d`, chọn `k` điểm, kiểm tra **khoảng quay vòng**; cắt tỉa số điểm xuất phát.
5. Vector hóa bằng `numpy` như một cách thay thế.

## 2. Đề bài & ví dụ

Hình vuông cạnh `side`, góc `(0,0)` và `(side,side)`. `points` là các điểm **phân biệt** nằm trên biên. Chọn `k` điểm sao cho **khoảng cách Manhattan nhỏ nhất** giữa hai điểm được chọn là **lớn nhất**. Trả về giá trị đó.

**Ví dụ 1:** `side = 2, points = [[0,2],[2,0],[2,2],[0,0]], k = 4` → `2`.

**Ví dụ 2:** `side = 2, points = [[0,0],[1,2],[2,0],[2,2],[2,1]], k = 4` → `1`.

**Ví dụ 3:** `side = 2, points = [[0,0],[0,1],[0,2],[1,2],[2,0],[2,2],[2,1]], k = 5` → `1`.

**Ràng buộc:** `1 ≤ side ≤ 10^9`, `4 ≤ |points| ≤ min(4·side, 15·10^3)`, `4 ≤ k ≤ min(25, |points|)`.

In [1]:
from typing import List
import itertools, bisect

examples = [
    (2, [[0, 2], [2, 0], [2, 2], [0, 0]], 4, 2),
    (2, [[0, 0], [1, 2], [2, 0], [2, 2], [2, 1]], 4, 1),
    (2, [[0, 0], [0, 1], [0, 2], [1, 2], [2, 0], [2, 2], [2, 1]], 5, 1),
]
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

for side, pts, k, exp in examples:
    print('side =', side, 'k =', k, '|points| =', len(pts), '-> kỳ vọng', exp)

side = 2 k = 4 |points| = 4 -> kỳ vọng 2
side = 2 k = 4 |points| = 5 -> kỳ vọng 1
side = 2 k = 5 |points| = 7 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tọa độ chu vi

Đi từ `(0,0)`: cạnh dưới (`y = 0`) → cạnh phải (`x = side`) → cạnh trên (`y = side`, `x` giảm) → cạnh trái (`x = 0`, `y` giảm). Mỗi điểm biên ứng với đúng một `t ∈ [0, 4·side)`.

In [2]:
def perim(side, x, y):
    if y == 0:
        return x                      # cạnh dưới
    if x == side:
        return side + y               # cạnh phải
    if y == side:
        return 3 * side - x           # cạnh trên
    return 4 * side - y               # cạnh trái (x == 0, y > 0)

side = 2
for p in [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (1, 2), (0, 2), (0, 1)]:
    print(p, '->', perim(side, *p))

(0, 0) -> 0
(1, 0) -> 1
(2, 0) -> 2
(2, 1) -> 3
(2, 2) -> 4
(1, 2) -> 5
(0, 2) -> 6
(0, 1) -> 7


### 3.2. Manhattan so với khoảng cách cung

Gọi `arc(p, q)` = khoảng cách ngắn nhất **dọc biên** (đi theo vòng, chọn chiều ngắn hơn).

- Cùng cạnh hoặc hai cạnh kề: `Manhattan = arc` (đường đi dọc biên qua tối đa một góc có dạng "chữ L").
- Hai cạnh đối: `Manhattan = side + |lệch|` ≤ `arc`, và luôn `≥ side`.

Tóm lại: **luôn `Manhattan ≤ arc`**, và nếu hai điểm ở hai cạnh đối thì `Manhattan ≥ side`.

In [3]:
import random

def arc(side, p, q):
    a, b = perim(side, *p), perim(side, *q)
    d = abs(a - b)
    return min(d, 4 * side - d)

def all_boundary(side):
    return sorted({(v, 0) for v in range(side + 1)} | {(side, v) for v in range(side + 1)} |
                  {(v, side) for v in range(side + 1)} | {(0, v) for v in range(side + 1)})

for s in range(1, 6):
    B = all_boundary(s)
    for p in B:
        for q in B:
            m, a = manhattan(p, q), arc(s, p, q)
            assert m <= a
            assert m == a or m >= s
print('Đã kiểm tra: Manhattan <= arc, và Manhattan < arc chỉ khi Manhattan >= side (side = 1..5).')

Đã kiểm tra: Manhattan <= arc, và Manhattan < arc chỉ khi Manhattan >= side (side = 1..5).


### 3.3. Tìm kiếm nhị phân đáp án

Nếu chọn được `k` điểm với khoảng cách nhỏ nhất ≥ `d` thì cũng được với mọi `d' < d` ⇒ hàm `ok(d)` **đơn điệu** ⇒ tìm `d` lớn nhất bằng nhị phân.

In [4]:
def max_true(lo, hi, ok):
    # giả sử ok(lo) đúng; tìm d lớn nhất trong [lo, hi] với ok(d)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(max_true(1, 100, lambda d: d * d <= 70))

8


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Cận trên `side`:** `k ≥ 4` điểm chia vòng dài `4·side` thành `k` cung, cung ngắn nhất `≤ 4·side / k ≤ side`; hai điểm đầu mút cung đó có `Manhattan ≤ arc ≤ side`. ⇒ Đáp án `≤ side`.

**Với `d ≤ side`:** một tập điểm có mọi cặp Manhattan ≥ `d` ⇔ mọi cặp **liền kề trên vòng** có cung ≥ `d`:

- (⇒) cung ≥ Manhattan ≥ `d`.
- (⇐) cặp bất kỳ: hai chiều đi vòng giữa chúng đều chứa ít nhất một cung liền kề nên `arc ≥ d`; nếu cùng cạnh/kề cạnh thì `Manhattan = arc ≥ d`, nếu hai cạnh đối thì `Manhattan ≥ side ≥ d`.

⇒ Bài toán trên hình vuông trở thành: **chọn `k` điểm trên một vòng tròn độ dài `4·side` sao cho các cung liền kề ≥ `d`**.

Ví dụ 2: tọa độ chu vi `[0, 2, 3, 4, 5]` (của `(0,0),(2,0),(2,1),(2,2),(1,2)`), chu vi 8, `k = 4`. Với `d = 2`: cần 4 điểm, các cung ≥ 2, tổng = 8 ⇒ mọi cung đúng bằng 2 ⇒ cần các vị trí cách đều 2 — chỉ có `0, 2, 4` (thiếu `6`) ⇒ không được. Với `d = 1`: được ⇒ đáp án 1.

In [5]:
side, pts, k, _ = examples[1]
print(sorted(perim(side, x, y) for x, y in pts))

[0, 2, 3, 4, 5]


## 5. Cách 1 — Brute Force: thử mọi tổ hợp `k` điểm

In [6]:
def max_dist_brute(side, pts, k):
    best = 0
    for comb in itertools.combinations(pts, k):
        best = max(best, min(manhattan(a, b) for a, b in itertools.combinations(comb, 2)))
    return best

for side, pts, k, exp in examples:
    print(max_dist_brute(side, pts, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


**Độ phức tạp:** `C(n, k) · k²` — chỉ dùng cho `n ≤ 10`.

## 6. Ý tưởng thuật toán / bước đột phá

**Hàm kiểm tra `ok(d)`** trên dãy `t` đã sắp, chu vi `P = 4·side`:

- Thử điểm xuất phát `i`. Tham lam: điểm kế tiếp là điểm **đầu tiên** có `t ≥ cur + d` (`bisect_left`). Lặp `k − 1` lần (không vượt quá cuối mảng).
- Thành công nếu chọn đủ `k` điểm và **cung quay vòng** `t[i] + P − cur ≥ d`.

**Vì sao tham lam đúng:** nếu có lời giải `p_1 < … < p_k` (theo vòng, bắt đầu từ `p_1`), tham lam bắt đầu tại `p_1` chọn các điểm `g_j ≤ p_j` (quy nạp) ⇒ điểm cuối `g_k ≤ p_k` ⇒ cung quay vòng của tham lam ≥ của lời giải ≥ `d`.

**Cắt tỉa điểm xuất phát:** lời giải có `k` cung ≥ `d` nên cung "chứa" `t[0]` dài ≤ `P − (k − 1)·d` ⇒ tồn tại điểm được chọn trong `[t[0], t[0] + P − (k − 1)·d]`; xuất phát từ điểm đó (coi nó là `p_1`) là đủ. ⇒ Chỉ thử `i` với `t[i] ≤ t[0] + P − (k−1)·d`.

Nhị phân `d ∈ [1, side]` (điểm phân biệt ⇒ `d = 1` luôn được).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Nhị phân + tham lam từ **mọi** điểm xuất phát

In [7]:
def max_dist_all_starts(side, pts, k):
    t = sorted(perim(side, x, y) for x, y in pts)
    n, P = len(t), 4 * side

    def ok(d):
        for i in range(n):
            cur, j, cnt = t[i], i, 1
            while cnt < k:
                j = bisect.bisect_left(t, cur + d, j + 1)
                if j >= n:
                    break
                cur, cnt = t[j], cnt + 1
            if cnt == k and t[i] + P - cur >= d:
                return True
        return False

    return max_true(1, side, ok)

for side, pts, k, exp in examples:
    print(max_dist_all_starts(side, pts, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


`O(log side · n · k · log n)` — ở cỡ lớn nhất khoảng `30 · 15000 · 25 · 14 ≈ 1,6·10^8` bước: chậm trong Python khi `ok(d)` sai (phải thử hết điểm xuất phát).

### 7.2. Cách 3 — Như cách 2 nhưng cắt tỉa điểm xuất phát (lời giải chính)

In [8]:
def max_dist_pruned(side, pts, k):
    t = sorted(perim(side, x, y) for x, y in pts)
    n, P = len(t), 4 * side

    def ok(d):
        limit = t[0] + P - (k - 1) * d
        for i in range(n):
            if t[i] > limit:
                break
            cur, j, cnt = t[i], i, 1
            while cnt < k:
                j = bisect.bisect_left(t, cur + d, j + 1)
                if j >= n:
                    break
                cur, cnt = t[j], cnt + 1
            if cnt == k and t[i] + P - cur >= d:
                return True
        return False

    return max_true(1, side, ok)

for side, pts, k, exp in examples:
    print(max_dist_pruned(side, pts, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


### 7.3. Cách 4 — Vector hóa bằng `numpy` (mọi điểm xuất phát cùng lúc)

Nhân đôi dãy (`t` và `t + P`) để khỏi xử lý vòng. `nxt[j] = searchsorted(ext, ext[j] + d)`. Bắt đầu `cur = [0..n−1]`, nhảy `k − 1` lần `cur = nxt[cur]` cho **tất cả** điểm xuất phát một lúc. Thành công nếu `cur < i + n` và `t[i] + P − ext[cur] ≥ d`.

In [9]:
import numpy as np

def max_dist_numpy(side, pts, k):
    t = np.array(sorted(perim(side, x, y) for x, y in pts), dtype=np.int64)
    n, P = len(t), 4 * side
    ext = np.concatenate([t, t + P])
    idx = np.arange(n)

    def ok(d):
        nxt = np.minimum(np.searchsorted(ext, ext + d), 2 * n - 1)   # kẹp chỉ số (kẹp => tự thất bại ở kiểm tra cuối)
        cur = idx.copy()
        for _ in range(k - 1):
            cur = nxt[cur]
        good = (cur < idx + n) & (t + P - ext[cur] >= d)
        return bool(good.any())

    return max_true(1, side, ok)

for side, pts, k, exp in examples:
    print(max_dist_numpy(side, pts, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


Vì sao "kẹp" an toàn: nếu một bước nhảy vượt quá mảng mở rộng thì `ext[cur] = ext[2n − 1] = t[n−1] + P ≥ t[i] + P` ⇒ `t[i] + P − ext[cur] ≤ 0 < d` ⇒ tự đánh dấu thất bại.

## 8. TRACE từng bước

Trace nhị phân + hàm kiểm tra cách 3 trên ví dụ 3.

In [10]:
def trace(side, pts, k):
    t = sorted(perim(side, x, y) for x, y in pts)
    n, P = len(t), 4 * side
    print('tọa độ chu vi:', t, '| P =', P)

    def ok(d):
        limit = t[0] + P - (k - 1) * d
        for i in range(n):
            if t[i] > limit:
                print(f"   d={d}: dừng thử xuất phát vì t[{i}]={t[i]} > giới hạn {limit}")
                break
            chosen, cur, j = [t[i]], t[i], i
            while len(chosen) < k:
                j = bisect.bisect_left(t, cur + d, j + 1)
                if j >= n:
                    break
                cur = t[j]; chosen.append(cur)
            wrap = t[i] + P - cur
            res = len(chosen) == k and wrap >= d
            print(f"   d={d}: xuất phát {t[i]} chọn {chosen}, cung quay vòng {wrap} -> {'OK' if res else 'không'}")
            if res:
                return True
        return False

    lo, hi = 1, side
    while lo < hi:
        mid = (lo + hi + 1) // 2
        r = ok(mid)
        print(f"thử d = {mid}: {'được' if r else 'không'}")
        if r: lo = mid
        else: hi = mid - 1
    print('đáp án:', lo)

trace(*examples[2][:3])

tọa độ chu vi: [0, 2, 3, 4, 5, 6, 7] | P = 8
   d=2: xuất phát 0 chọn [0, 2, 4, 6], cung quay vòng 2 -> không
   d=2: dừng thử xuất phát vì t[1]=2 > giới hạn 0
thử d = 2: không
đáp án: 1


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def maxDistance(self, side: int, points: List[List[int]], k: int) -> int:
        def perim(x, y):
            if y == 0:
                return x
            if x == side:
                return side + y
            if y == side:
                return 3 * side - x
            return 4 * side - y

        t = sorted(perim(x, y) for x, y in points)
        n, P = len(t), 4 * side

        def ok(d):
            limit = t[0] + P - (k - 1) * d
            for i in range(n):
                if t[i] > limit:
                    break
                cur, j, cnt = t[i], i, 1
                while cnt < k:
                    j = bisect.bisect_left(t, cur + d, j + 1)
                    if j >= n:
                        break
                    cur, cnt = t[j], cnt + 1
                if cnt == k and t[i] + P - cur >= d:
                    return True
            return False

        lo, hi = 1, side
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

for side, pts, k, exp in examples:
    got = Solution().maxDistance(side, pts, k)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
1 OK


## 10. Vì sao thuật toán đúng

1. **Đáp án trong `[1, side]`:** điểm phân biệt ⇒ Manhattan ≥ 1; `k ≥ 4` ⇒ ≤ `side` (mục 4).
2. **Tương đương vòng tròn với `d ≤ side`:** chọn được `k` điểm có mọi Manhattan ≥ `d` ⇔ chọn được `k` điểm có mọi cung liền kề (kể cả quay vòng) ≥ `d` (mục 4, đã kiểm chứng bằng máy ở 3.2).
3. **Đơn điệu** ⇒ nhị phân đúng.
4. **Hàm kiểm tra đúng:** nếu nó trả `True` thì tập chọn thỏa (các cung liền kề ≥ `d` theo cách chọn và kiểm tra quay vòng). Nếu có lời giải, gọi `q` là điểm được chọn đầu tiên tính từ `t[0]` theo chiều tăng; cung chứa `t[0]` (từ điểm chọn cuối tới `q`) ≤ `P − (k − 1)d` ⇒ `q ≤ t[0] + P − (k−1)d` nên `q` nằm trong vùng thử; tham lam bắt đầu tại `q` chọn các điểm không muộn hơn lời giải ⇒ thành công.

## 11. Phân tích Time / Space Complexity

Gọi `n = |points| ≤ 15000`, `L = log₂ side ≤ 30`.

| Cách | Time | Space |
|---|---|---|
| 1. Tổ hợp | `C(n, k) · k²` | `O(k)` |
| 2. Nhị phân + mọi xuất phát | `O(L · n · k · log n)` | `O(n)` |
| 3. Nhị phân + cắt tỉa | tệ nhất như 2, thực tế nhỏ hơn nhiều | `O(n)` |
| 4. `numpy` | `O(L · (n log n + n k))` thao tác vector | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'all_starts': max_dist_all_starts, 'pruned': max_dist_pruned, 'numpy': max_dist_numpy,
         'submit': lambda s, p, k: Solution().maxDistance(s, p, k)}
edge = [
    (1, [[0, 0], [1, 0], [1, 1], [0, 1]], 4, 1),
    (3, [[0, 0], [3, 0], [3, 3], [0, 3]], 4, 3),            # bốn góc: đúng bằng side
    (4, [[2, 0], [4, 2], [2, 4], [0, 2]], 4, 4),            # trung điểm các cạnh
    (4, [[0, 0], [1, 0], [2, 0], [3, 0], [4, 0], [4, 4]], 4, 2),   # chọn (0,0),(2,0),(4,0),(4,4)
]
for side, pts, k, exp in examples + edge:
    assert max_dist_brute(side, pts, k) == exp, (side, pts, k, max_dist_brute(side, pts, k))
    for name, f in impls.items():
        assert f(side, pts, k) == exp, (name, side, pts, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
def rand_points(side, m, rng):
    s = set()
    while len(s) < m:
        v = rng.randint(0, side)
        s.add(rng.choice([(v, 0), (side, v), (v, side), (0, v)]))
    return [list(p) for p in s]

rng = random.Random(3464)
cnt = 0
for _ in range(1500):
    side = rng.randint(1, 7)
    m = rng.randint(4, min(4 * side, 9))
    k = rng.randint(4, m)
    pts = rand_points(side, m, rng)
    exp = max_dist_brute(side, pts, k)
    for name, f in impls.items():
        assert f(side, pts, k) == exp, (name, side, pts, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 1500 bộ ngẫu nhiên.


### 12.3. Ca lớn: `side = 10^9`, `15000` điểm

In [14]:
import time
rng = random.Random(7)
big_rand = rand_points(10 ** 9, 15000, rng)
dense_side = 3750
dense = [list(p) for p in all_boundary(dense_side)]          # 15000 điểm: phủ kín biên
cluster = [[v, 0] for v in range(0, 14000)] + [[10 ** 9, 10 ** 9], [0, 10 ** 9], [10 ** 9, 0]]
for name, side, pts in [('ngẫu nhiên', 10 ** 9, big_rand), ('phủ kín biên', dense_side, dense), ('dồn một cạnh', 10 ** 9, cluster)]:
    for k in (4, 25):
        t0 = time.time(); a = Solution().maxDistance(side, pts, k); t1 = time.time() - t0
        t0 = time.time(); b = max_dist_numpy(side, pts, k); t2 = time.time() - t0
        assert a == b
        print(f"{name:13s} n={len(pts):5d} k={k:2d}: {a:10d} | cắt tỉa {t1:.2f}s | numpy {t2:.2f}s")

t0 = time.time(); c = max_dist_all_starts(10 ** 9, cluster, 25); t3 = time.time() - t0
print(f"cách 2 (không cắt tỉa) trên 'dồn một cạnh', k=25: {c} | {t3:.2f}s")

ngẫu nhiên    n=15000 k= 4:  999989926 | cắt tỉa 0.04s | numpy 0.02s
ngẫu nhiên    n=15000 k=25:  159889178 | cắt tỉa 0.03s | numpy 0.02s
phủ kín biên  n=15000 k= 4:       3750 | cắt tỉa 0.00s | numpy 0.01s
phủ kín biên  n=15000 k=25:        600 | cắt tỉa 0.01s | numpy 0.01s
dồn một cạnh  n=14003 k= 4: 1000000000 | cắt tỉa 0.00s | numpy 0.01s


dồn một cạnh  n=14003 k=25:        666 | cắt tỉa 0.27s | numpy 0.03s


cách 2 (không cắt tỉa) trên 'dồn một cạnh', k=25: 666 | 0.33s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định, so với vét cạn):

In [15]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        side = rng.randint(1, 5)
        m = rng.randint(4, min(4 * side, 8))
        k = rng.randint(4, m)
        pts = rand_points(side, m, rng)
        if wrong(side, pts, k) != max_dist_brute(side, pts, k):
            return side, pts, k
    return None

def make_solver(check_wrap=True, only_first=False, perim_fn=perim):
    def solve(side, pts, k):
        t = sorted(perim_fn(side, x, y) for x, y in pts)
        n, P = len(t), 4 * side
        def ok(d):
            for i in range(1 if only_first else n):
                cur, j, cnt = t[i], i, 1
                while cnt < k:
                    j = bisect.bisect_left(t, cur + d, j + 1)
                    if j >= n:
                        break
                    cur, cnt = t[j], cnt + 1
                if cnt == k and (not check_wrap or t[i] + P - cur >= d):
                    return True
            return False
        return max_true(1, side, ok)
    return solve

### Lỗi 1 — Quên kiểm tra cung quay vòng (điểm cuối với điểm đầu)

In [16]:
wrong_no_wrap = make_solver(check_wrap=False)
ce = find_counterexample(wrong_no_wrap, seed=1)
print('phản ví dụ:', ce)
print('sai :', wrong_no_wrap(*ce), '| đúng:', max_dist_brute(*ce))

phản ví dụ: (4, [[0, 1], [4, 4], [0, 4], [3, 4], [1, 0]], 4)
sai : 3 | đúng: 2


**Vì sao sai:** biên hình vuông là **vòng khép kín**; điểm cuối và điểm đầu cũng là hai điểm liền kề, có thể quá gần nhau.

### Lỗi 2 — Chỉ xuất phát từ điểm có tọa độ chu vi nhỏ nhất

In [17]:
wrong_first_only = make_solver(only_first=True)
ce = find_counterexample(wrong_first_only, seed=2)
print('phản ví dụ:', ce)
print('sai :', wrong_first_only(*ce), '| đúng:', max_dist_brute(*ce))

phản ví dụ: (3, [[0, 1], [0, 0], [2, 0], [3, 0], [0, 2], [3, 3], [3, 2], [1, 3]], 4)
sai : 2 | đúng: 3


**Vì sao sai:** trên đường thẳng, xuất phát từ điểm đầu là tối ưu; trên **vòng**, điểm `t[0]` có thể không thuộc lời giải tốt nhất (nó "ăn" mất cung quay vòng).

### Lỗi 3 — Duỗi biên sai thứ tự (cạnh trên/trái không đảo chiều)

In [18]:
def perim_wrong(side, x, y):
    if y == 0:
        return x
    if x == side:
        return side + y
    if y == side:
        return 2 * side + x            # SAI: phải là 3*side - x
    return 3 * side + y                # SAI: phải là 4*side - y

wrong_perim = make_solver(perim_fn=perim_wrong)
ce = find_counterexample(wrong_perim, seed=3)
print('phản ví dụ:', ce)
print('sai :', wrong_perim(*ce), '| đúng:', max_dist_brute(*ce))

phản ví dụ: (2, [[0, 1], [1, 2], [0, 0], [2, 0], [0, 2], [2, 2], [1, 0]], 4)
sai : 1 | đúng: 2


**Vì sao sai:** tọa độ chu vi phải **liên tục** khi đi quanh biên; nếu cạnh trên đi theo `x` tăng thì `(side, side)` và `(0, side)` bị đảo vị trí, khoảng cách cung không còn phản ánh đúng.

### Lỗi 4 — Dùng khoảng cách cung làm đáp án mà quên cận trên `side`

In [19]:
def wrong_upper(side, pts, k):
    t = sorted(perim(side, x, y) for x, y in pts)
    n, P = len(t), 4 * side
    def ok(d):
        for i in range(n):
            cur, j, cnt = t[i], i, 1
            while cnt < k:
                j = bisect.bisect_left(t, cur + d, j + 1)
                if j >= n:
                    break
                cur, cnt = t[j], cnt + 1
            if cnt == k and t[i] + P - cur >= d:
                return True
        return False
    return max_true(1, 4 * side, ok)               # SAI nếu k nhỏ hơn 4 (đề chặn k >= 4)

# Với k >= 4 (đúng đề) cận 4*side cho cùng kết quả — minh họa vì sao điều kiện k >= 4 quan trọng:
pts2 = [[0, 1], [4, 1]]                         # hai điểm ở hai cạnh đối, side = 4
print('k = 2 (ngoài đề): cung =', wrong_upper(4, pts2, 2), 'nhưng Manhattan thật =', manhattan(*pts2))
print('k >= 4: cùng kết quả?', all(wrong_upper(s, p, k) == Solution().maxDistance(s, p, k) for s, p, k, _ in examples))

k = 2 (ngoài đề): cung = 6 nhưng Manhattan thật = 4
k >= 4: cùng kết quả? True


**Vì sao cần chú ý:** khi `d > side` thì "cung ≥ d" **không** còn kéo theo "Manhattan ≥ d" (hai điểm ở cạnh đối có Manhattan nhỏ hơn cung). Lập luận của bài dựa trên `k ≥ 4` ⇒ đáp án ≤ `side`. Nếu đề đổi `k` có thể bằng 2 hoặc 3 thì cách quy về vòng tròn phải sửa.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa hóa khoảng cách nhỏ nhất" ⇒ **nhị phân đáp án** + hàm kiểm tra tham lam.
- Điểm trên **biên đa giác/hình vuông** ⇒ duỗi thành vòng tròn bằng tọa độ chu vi.
- Vòng tròn ⇒ nhân đôi mảng hoặc kiểm tra cung quay vòng, và thử nhiều điểm xuất phát (có thể cắt tỉa).
- Khi ràng buộc nhỏ (`k ≤ 25`) nhưng `n` lớn ⇒ nhảy `k` lần bằng `bisect` từ mỗi điểm xuất phát.

## 15. Biến thể của bài

1. **LC 1552 — Magnetic Force Between Two Balls:** cùng ý tưởng trên **đường thẳng** (không vòng).
2. **LC 2517 — Maximum Tastiness of Candy Basket:** nhị phân + tham lam.
3. Điểm trên **đường tròn** (khoảng cách cung) ⇒ chính là phần lõi của bài này.
4. Đổi sang khoảng cách Euclid ⇒ không còn tương đương với cung đơn giản.

In [20]:
# Biến thể 3: k điểm trên vòng tròn chu vi P, tối đa hóa cung nhỏ nhất (dùng lại lõi của bài)
def max_min_arc(P, positions, k):
    t = sorted(positions)
    n = len(t)
    def ok(d):
        limit = t[0] + P - (k - 1) * d
        for i in range(n):
            if t[i] > limit:
                break
            cur, j, cnt = t[i], i, 1
            while cnt < k:
                j = bisect.bisect_left(t, cur + d, j + 1)
                if j >= n:
                    break
                cur, cnt = t[j], cnt + 1
            if cnt == k and t[i] + P - cur >= d:
                return True
        return False
    return max_true(0, P, ok)

def max_min_arc_brute(P, positions, k):
    best = 0
    for c in itertools.combinations(sorted(positions), k):
        gaps = [c[i + 1] - c[i] for i in range(k - 1)] + [c[0] + P - c[-1]]
        best = max(best, min(gaps))
    return best

rng = random.Random(15)
for _ in range(500):
    P = rng.randint(5, 30)
    pos = rng.sample(range(P), rng.randint(2, min(P, 8)))
    k = rng.randint(2, len(pos))
    assert max_min_arc(P, pos, k) == max_min_arc_brute(P, pos, k)
print('Biến thể vòng tròn khớp vét cạn:', max_min_arc(12, [0, 1, 4, 7, 8, 10], 3))

Biến thể vòng tròn khớp vét cạn: 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Tổ hợp | mũ | Oracle |
| 2 | Nhị phân + mọi xuất phát | `O(L n k log n)` | Đúng, có thể chậm |
| 3 | Nhị phân + cắt tỉa xuất phát | thực tế nhanh | **Nộp bài** |
| 4 | `numpy` vector hóa | nhanh, ổn định | Cần `numpy` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `side = 5`, bốn góc, `k = 4`.

**Bài 2 (tính tay):** `side = 4`, `points = [[0,0],[2,0],[4,0],[4,2],[4,4],[2,4],[0,4],[0,2]]`, `k = 8`.

**Bài 3 (điền TODO):** viết `magnetic(positions, m)` — LC 1552 trên đường thẳng.

**Bài 4 (tự viết lại):** trả về **chính các điểm** được chọn (không chỉ khoảng cách).

**Bài 5 (mini challenge):** kiểm tra bằng máy rằng với `k = 3` cách quy về vòng tròn có thể sai (tìm phản ví dụ).

In [21]:
def magnetic_todo(positions, m):
    # TODO
    raise NotImplementedError

def chosen_points_todo(side, points, k):
    # TODO
    raise NotImplementedError

def find_k3_counterexample_todo():
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Bốn góc cách nhau `side` theo cung. (2) Cặp góc đối: Manhattan `2·side`. (3) Min là `side`.

**Bài 2.** (1) 8 điểm cách đều 2 trên chu vi 16. (2) Chọn hết. (3) Min Manhattan giữa hai điểm liền kề.

**Bài 3.** (1) Sắp xếp. (2) Nhị phân khoảng cách. (3) Tham lam từ điểm đầu (không cần vòng).

**Bài 4.** (1) Chạy lại hàm kiểm tra với `d` tối ưu. (2) Lưu danh sách tọa độ chu vi được chọn. (3) Đổi ngược về điểm `(x, y)` bằng từ điển.

**Bài 5.** (1) Dùng lõi vòng tròn với `k = 3`, nhưng không chặn `d ≤ side`. (2) So với vét cạn Manhattan. (3) Duyệt `side` nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
a1 = Solution().maxDistance(5, [[0, 0], [5, 0], [5, 5], [0, 5]], 4)
b2 = [[0, 0], [2, 0], [4, 0], [4, 2], [4, 4], [2, 4], [0, 4], [0, 2]]
a2 = Solution().maxDistance(4, b2, 8)
assert a1 == 5 == max_dist_brute(5, [[0, 0], [5, 0], [5, 5], [0, 5]], 4)
assert a2 == 2 == max_dist_brute(4, b2, 8)
print('Bài 1:', a1, '| Bài 2:', a2)

# Bài 3
def magnetic(positions, m):
    p = sorted(positions)
    def ok(d):
        cnt, last = 1, p[0]
        for x in p[1:]:
            if x - last >= d:
                cnt += 1; last = x
        return cnt >= m
    return max_true(1, p[-1] - p[0], ok)

assert magnetic([1, 2, 3, 4, 7], 3) == 3 and magnetic([5, 4, 3, 2, 1, 1000000000], 2) == 999999999
print('Bài 3: LC 1552 ví dụ ->', magnetic([1, 2, 3, 4, 7], 3), magnetic([5, 4, 3, 2, 1, 1000000000], 2))

Bài 1: 5 | Bài 2: 2
Bài 3: LC 1552 ví dụ -> 3 999999999


In [23]:
# Bài 4
def chosen_points(side, points, k):
    best = Solution().maxDistance(side, points, k)
    back = {perim(side, x, y): [x, y] for x, y in points}
    t = sorted(back)
    n, P = len(t), 4 * side
    for i in range(n):
        sel, cur, j = [t[i]], t[i], i
        while len(sel) < k:
            j = bisect.bisect_left(t, cur + best, j + 1)
            if j >= n:
                break
            cur = t[j]; sel.append(cur)
        if len(sel) == k and t[i] + P - cur >= best:
            return best, [back[v] for v in sel]
    return best, []

rng = random.Random(4)
for _ in range(300):
    side = rng.randint(1, 8); m = rng.randint(4, min(4 * side, 12)); k = rng.randint(4, m)
    pts = rand_points(side, m, rng)
    d, sel = chosen_points(side, pts, k)
    assert len(sel) == k and min(manhattan(a, b) for a, b in itertools.combinations(sel, 2)) == d
print('Bài 4 hợp lệ trên 300 bộ; ví dụ 2:', chosen_points(*examples[1][:3]))

Bài 4 hợp lệ trên 300 bộ; ví dụ 2: (1, [[0, 0], [2, 0], [2, 1], [2, 2]])


In [24]:
# Bài 5 — với k = 3, cách quy về vòng tròn (không chặn d <= side) có thể sai
def circle_method_any_k(side, pts, k):
    t = [perim(side, x, y) for x, y in pts]
    return max_min_arc(4 * side, t, k)

found = None
for side in range(1, 5):
    B = all_boundary(side)
    for comb in itertools.combinations(B, 3):
        pts = [list(p) for p in comb]
        if circle_method_any_k(side, pts, 3) != max_dist_brute(side, pts, 3):
            found = (side, pts); break
    if found:
        break
print('phản ví dụ k = 3:', found, '| vòng tròn:', circle_method_any_k(found[0], found[1], 3),
      '| Manhattan thật:', max_dist_brute(found[0], found[1], 3))

phản ví dụ k = 3: (4, [[0, 1], [2, 4], [4, 1]]) | vòng tròn: 5 | Manhattan thật: 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
t = sorted(perim(x, y))            # dưới: x; phải: side+y; trên: 3side-x; trái: 4side-y
def ok(d):
    for i trong khi t[i] <= t[0] + 4side - (k-1)d:
        nhảy k-1 lần: j = bisect_left(t, cur + d, j + 1)
        nếu đủ k điểm và t[i] + 4side - cur >= d: return True
    return False
nhị phân d trong [1, side]
```

**Checklist:**

- [ ] Tọa độ chu vi liên tục (cạnh trên, cạnh trái đi ngược)?
- [ ] Kiểm tra cung quay vòng?
- [ ] Thử nhiều điểm xuất phát (có cắt tỉa)?
- [ ] Cận trên nhị phân `side` (dựa trên `k ≥ 4`)?
- [ ] `bisect_left` bắt đầu từ `j + 1` để không chọn lại điểm cũ?